# LLDP demo 能力

## LLDP 是什么
LLDP（Link Layer Discovery Protocol） 是一种网络层协议

主要用于在同一局域网内的网络设备之间发现彼此的身份和拓扑结构信息。
LLDP 是 IEEE 802.1AB 标准的一部分，它允许网络设备（如交换机、路由器、无线接入点、服务器等）在数据链路层交换信息，以便彼此了解网络拓扑、设备信息、接口详情等

因此可以基于LLDP协议获取网络的物理拓扑关系，获取集群的交换机、服务器、rank关系


## 环境搭建
使用scapy包，基于此获取集群信息

In [ ]:
pip install scapy

## 捕获LLDP包
通过scapy，启动一个监听的进程，捕获LLDP包
注意，一台服务器上有多个可被监听的接口。
但是有些是localhost、docker等，这些接口无法捕获到LLDP包
需要找到对外的接口

In [ ]:
from scapy.all import sniff, Ether, raw

# 回调函数用于处理捕获的数据包
def lldp_packet_handler(packet):
    # 检查数据包是否是以太网数据包并且包含 LLDP 类型
    if packet.haslayer(Ether) and packet.type == 0x88CC:  # 0x88CC 是 LLDP 的以太网类型
        print("Captured LLDP packet:")
        packet.show()

# 捕获网络接口上的 LLDP 数据包
# sniff(iface="你的网络接口", filter="ether proto 0x88CC", prn=lldp_packet_handler, store=0)

sniff(iface="enp189s0f0", filter="ether proto 0x88CC", prn=lldp_packet_handler, store=0)

如果出现 ValueError: Interface 'xxxx' not found !
或者进程启动一分钟后无捕获到的数据（LLDP通常30s/60s会发送一次）

说明接口选择错误
（也有可能是设备未启动LLDP，假设集群环境的设备已启动LLDP）

可以使用以下方式查看本地有哪些接口
优先尝试enp开头的接口

In [ ]:
ip link show

In [ ]:
from scapy.all import get_if_list
print(get_if_list())